# Best Buy Canada Review Sentiment Analysis

This notebook documents the collection, validation, sentiment-scoring, export, and stakeholder-insight workflow. Collection is **off by default**. No Best Buy data is included or invented. Before running a collection, check the current terms and `https://www.bestbuy.ca/robots.txt`; use only direct product pages and visible public review controls. Never bypass a block, CAPTCHA, or rate limit.

The live terms page and live product review markup could not be verified while preparing this project. A scrape of at least 50 reviews and product-specific findings therefore remain unverified.

## 1. Setup and shared application functions

Install the dependencies from `requirements.txt` in the active Python environment before running the cells.

In [ ]:
from pathlib import Path
import pandas as pd

from src.bestbuy_reviews import ScrapeSettings, scrape_reviews
from src.data_io import load_review_csv
from src.sentiment import analyze_sentiment, build_insights, recommendations

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
print('Ready. Scraping is not started by importing the modules.')

## 2. Optional public review collection

Set `RUN_SCRAPE = True` only after confirming the current site terms, robots.txt permission, and your authorization. The collector restricts URLs to `/en-ca/product/`, checks robots.txt before browsing, uses visible controls, applies a polite pause and maximum page count, and will stop if the visible layout is not recognized. It does not access private APIs or circumvent access controls.

In [ ]:
PRODUCT_URL = 'https://www.bestbuy.ca/en-ca/product/samsung-5-2-cu-ft-iec-front-load-washer-with-smart-care-and-self-clean-energy-star-wf45t6000aw-a5-white/14400381'
SORT_VIEWS = ['Most Helpful', 'Newest', 'Highest Rating', 'Lowest Rating', 'Most Relevant']
MAX_PAGES_PER_SORT = 6
RUN_SCRAPE = False  # Change only after reviewing current policies and permissions
POLICY_CONFIRMED = False  # Explicit operator confirmation is required

scraped_reviews = pd.DataFrame()
if RUN_SCRAPE:
    if not POLICY_CONFIRMED:
        raise PermissionError('Confirm current terms, robots.txt, and authorization before scraping.')
    scraped_reviews = scrape_reviews(
        ScrapeSettings(
            product_url=PRODUCT_URL,
            sort_options=tuple(SORT_VIEWS),
            max_pages_per_sort=MAX_PAGES_PER_SORT,
            pause_seconds=2.0,
        ),
        policy_confirmed=POLICY_CONFIRMED,
    )
    print(f'Collected {len(scraped_reviews)} unique reviews. Check whether this meets the 50-review minimum.')
else:
    print('Collection skipped. Provide an authorized CSV below to analyze real review data.')

## 3. Load and validate review data

Use the scrape result or a UTF-8 CSV you are authorized to analyze. The required text field is `review_text`; expected metadata fields include `primary_key`, `title`, `date`, `rating`, `source`, and `reviewer_name`. Do not treat synthetic examples as scraped output.

In [ ]:
CSV_PATH = Path('outputs/reviews.csv')  # Set to your authorized input CSV if needed
if not scraped_reviews.empty:
    reviews = scraped_reviews.copy()
elif CSV_PATH.exists():
    reviews = load_review_csv(CSV_PATH)
else:
    reviews = pd.DataFrame()

if not reviews.empty:
    if 'review_text' not in reviews.columns:
        raise ValueError('Input CSV must contain a review_text column.')
    reviews['review_text'] = reviews['review_text'].fillna('').astype(str)
    reviews = reviews.loc[reviews['review_text'].str.strip().ne('')].copy()
    print(f'{len(reviews)} usable reviews loaded; 50-review threshold met: {len(reviews) >= 50}')
else:
    print('No authorized review data found. Set CSV_PATH or run the opt-in collection cell.')

## 4. Sentiment analysis and review categories

VADER produces a compound score from -1 to +1. The shared code labels scores >= 0.05 Positive, <= -0.05 Negative, and the remainder Neutral. It also combines sentiment with the star rating to create interpretable categories. This is an exploratory English-language model; validate a hand-coded sample before decisions.

In [ ]:
if not reviews.empty:
    analyzed = analyze_sentiment(reviews)
    display(analyzed.head(10))
    print(analyzed['sentiment'].value_counts(dropna=False))
else:
    analyzed = pd.DataFrame()
    print('Sentiment scoring skipped until review data is supplied.')

## 5. Export CSV and Excel results

Exports are generated only from supplied or collected data. No empty or synthetic dataset is presented as a Best Buy scrape.

In [ ]:
if not analyzed.empty:
    insights = build_insights(analyzed)
    csv_output = OUTPUT_DIR / 'bestbuy_reviews_sentiment.csv'
    excel_output = OUTPUT_DIR / 'bestbuy_reviews_sentiment.xlsx'
    analyzed.to_csv(csv_output, index=False, encoding='utf-8-sig')
    with pd.ExcelWriter(excel_output, engine='openpyxl') as writer:
        analyzed.to_excel(writer, index=False, sheet_name='Reviews')
        pd.DataFrame([{'metric': k, 'value': str(v)} for k, v in insights.items()]).to_excel(
            writer, index=False, sheet_name='Insights'
        )
    print(f'Wrote {csv_output} and {excel_output}')
else:
    print('No export created.')

## 6. Stakeholder insight summary

Term counts are hypotheses, not causal explanations. Validate themes with review reading and operational data (returns, support contacts, product variants) before recommending a product change.

In [ ]:
if not analyzed.empty:
    insights = build_insights(analyzed)
    print(f"Reviews: {insights['review_count']}")
    print(f"Average rating: {insights['average_rating']}")
    print(f"Positive share: {insights['positive_share']:.1%}")
    print(f"Negative share: {insights['negative_share']:.1%}")
    print('Potential positive themes:', insights['top_positive_terms'])
    print('Potential negative themes:', insights['top_negative_terms'])
    for action in recommendations(insights):
        print('-', action)
else:
    print('No data available for stakeholder insights.')

## Limitations and responsible use

- Re-check Best Buy robots.txt and terms immediately before any run. If the page or policy is unclear, stop and request permission.
- The collector uses the site's visible Reject All option to decline optional cookies when available; it never accepts optional cookies automatically. If that option is missing, collection stops; this app does not provide a manual consent flow for its browser session. Do not use proxy rotation, CAPTCHA bypass, fake identities, login automation, or rate-limit evasion.
- Selenium requires a compatible Chrome browser. The page's visible labels and markup may change; errors indicate that collection is incomplete, not that the product has no reviews.
- Deduplication across sort views and maximum pages can leave a partial sample. Compare the output with the public count; the app warns when fewer than 50 reviews are available.
- Reviewer names may be personal data. Minimize, pseudonymize, and retain only as long as needed.
- See `reports/stakeholder_report.md` for scraping challenges and stakeholder recommendations.